# NANO-GPT 150M — мультидиалоговый корпус
d_model 1024, 12 слоёв, 16 голов, контекст 512, fp16 (AMP).
Корпус: 95 млн символов — ru_turbo_saiga (37.7к мультидиалогов) + saiga_scored (22.5к) + OASST ru (721 дерево) + наш рольплей-датасет (131к пар).
Мультидиалоги = модель учится связывать реплики в тему.
~4-8 часов на T4, чекпоинты каждые 500 шагов сразу на Google Drive.

Порядок: выполни ячейки сверху вниз. Драйв монтируем ДО обучения.

In [ ]:
# Клон и проверка GPU
!git clone https://github.com/lucifermornngstar52-cell/nano-gpt.git
%cd nano-gpt
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader || echo 'GPU НЕ ВКЛЮЧЕНА!'

In [ ]:
# Смонтировать Drive ДО обучения — чекпоинты падают сразу туда
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nano-gpt-150m

In [ ]:
# Обучение 150M (~4-8 ч на T4)
import os
os.environ.update(
    DATA='corpus_big.txt.gz',      # мультидиалоговый корпус 95M символов
    DM='1024', NL='12', NH='16', CTX='512',
    STEPS='25000', BS='8',         # ~1 эпоха по корпусу
    CKPT='/content/drive/MyDrive/nano-gpt-150m',
)
%cd /content/nano-gpt
!python train_torch.py

In [ ]:
# Чат-тест (после обучения или в любой момент на свежем чекпоинте)
%cd /content/nano-gpt
import os, torch, numpy as np, json
os.environ.update(DM='1024', NL='12', NH='16', CTX='512')
from train_torch import NanoT, vocab, stoi, dev, NL, T, DM
model = NanoT().to(dev)
z = np.load('/content/drive/MyDrive/nano-gpt-150m/weights_big.npz')
with torch.no_grad():
    model.we.copy_(torch.tensor(z['we'], device=dev)); model.wpe.copy_(torch.tensor(z['wpe'], device=dev))
    for l in range(NL):
        b = model.blocks[l]
        for nm in ('q','k','v','o','fc','fc2'): getattr(b, nm).copy_(torch.tensor(z[f'{l}{nm}'], device=dev))
        b.ln1.weight.copy_(torch.tensor(z[f'{l}ln1g'], device=dev)); b.ln1.bias.copy_(torch.tensor(z[f'{l}ln1b'], device=dev))
        b.ln2.weight.copy_(torch.tensor(z[f'{l}ln2g'], device=dev)); b.ln2.bias.copy_(torch.tensor(z[f'{l}ln2b'], device=dev))
    model.lnf.weight.copy_(torch.tensor(z['lnfg'], device=dev)); model.lnf.bias.copy_(torch.tensor(z['lnfb'], device=dev))
@torch.no_grad()
def reply(q, temp=0.8):
    idx = torch.tensor([[stoi.get(c,0) for c in ('В: '+q+'\nО:')[-T:]]], device=dev)
    out = ''
    for _ in range(150):
        lg = model(idx[:, -T:])[0,-1]/temp
        k = torch.topk(lg, 8); ni = k.indices[torch.multinomial(torch.softmax(k.values,-1),1)]
        ch = vocab[ni.item()]; out += ch
        if ch=='\n' and out.endswith('\n\n'): break
        idx = torch.cat([idx, ni.view(1,1)],1)
    return out.strip()
for q in ['как дела?','привет','расскажи о себе','что ты умеешь?']:
    print('В:', q, '\nО:', reply(q), '\n')

In [ ]:
# Резюм, если обучение оборвалось.
# 1) подставь номер последнего чекпоинта вместо 5000 (смотри файлы в MyDrive/nano-gpt-150m и лог)
# 2) запусти эту ячейку — она вернёт веса с Drive в рабочую папку и продолжит с нужного шага
import os, shutil
START = 5000  # <- номер шага последнего сохранения
shutil.copy('/content/drive/MyDrive/nano-gpt-150m/weights_big.npz', '/content/nano-gpt/weights_big.npz')
os.environ.update(DATA='corpus_big.txt.gz', DM='1024', NL='12', NH='16', CTX='512',
                  STEPS='25000', BS='8', CKPT='/content/drive/MyDrive/nano-gpt-150m', START=str(START))
%cd /content/nano-gpt
!python train_torch.py